# Shor Algorithm Exercise: From QFT to Factoring

This notebook walks step-by-step through the core pieces of Shor's algorithm.
We begin with the **Quantum Fourier Transform (QFT)** and then build up to **period finding** and factorization of a small composite number.

> Suggested target: factor **N = 15** using base **a = 2**.

## Learning goals

By the end, you should be able to:

1. Build and visualize a QFT circuit.
2. Understand modular exponentiation values used in Shor's algorithm.
3. Build a period-finding circuit skeleton.
4. Extract a candidate period `r` from measured phases.
5. Use `gcd(a^(r/2) ± 1, N)` to recover factors.

In [ ]:
# If needed, install dependencies in your environment:
# !pip install qiskit qiskit-aer matplotlib

import math
from math import gcd
from fractions import Fraction

import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.circuit.library import QFT

# Try to load Aer simulator; if unavailable, raise a friendly message.
try:
    from qiskit_aer import AerSimulator
except Exception as exc:
    raise ImportError(
        "qiskit-aer is required for simulation. Install with: pip install qiskit-aer"
    ) from exc

## 1) Warm-up: QFT on 3 qubits

The QFT maps computational basis states into phase-encoded superpositions.
In Shor's algorithm, we apply an inverse QFT to reveal periodicity.

In [ ]:
n_qft = 3
qft_circuit = QFT(num_qubits=n_qft, do_swaps=True)
qft_circuit.draw('mpl')

### Exercise 1

- Change `n_qft` to 4 or 5.
- Set `do_swaps=False` and compare the diagram.
- Explain why swaps may be optional depending on bit-order postprocessing.

## 2) Pick a factoring instance

Shor factors a composite `N` by finding the order (period) `r` of `a mod N`, i.e. the smallest positive `r` such that:

\[
a^r \equiv 1 \; (\text{mod } N)
\]

We need `gcd(a, N)=1`.

In [ ]:
N = 15
a = 2

assert gcd(a, N) == 1, "a and N must be coprime"

# Classical preview of modular powers
values = [pow(a, x, N) for x in range(16)]
print("a^x mod N for x=0..15:")
print(values)

# Find the true period classically (for verification)
r_true = None
for r in range(1, 2*N):
    if pow(a, r, N) == 1:
        r_true = r
        break
print(f"True period (for reference): r = {r_true}")

### Exercise 2

- Try `a = 7` with `N = 15`.
- Observe how the sequence changes and verify the period.

## 3) Build a period-finding circuit (N=15, a=2 demo)

For pedagogical clarity, we use a compact demonstration circuit with:
- counting register size `t`
- work register size `n = ceil(log2(N))`

A full scalable implementation needs modular exponentiation unitaries. Here, we focus on the key structure and phase-estimation behavior.

In [ ]:
t = 4  # counting qubits
n = math.ceil(math.log2(N))  # work qubits

qc = QuantumCircuit(t + n, t)

# 1) Prepare counting register in uniform superposition
for q in range(t):
    qc.h(q)

# 2) Initialize work register to |1>
qc.x(t)  # least-significant work qubit as |1>

# 3) Controlled-U^(2^k) blocks
# In a full implementation, each block applies modular multiplication by a^(2^k) mod N.
# For this exercise notebook, we emulate phase kickback behavior for N=15, a=2 using controlled phase rotations.
for k in range(t):
    angle = 2 * np.pi * (2**k) / r_true
    qc.cp(angle, k, t)

# 4) Inverse QFT on counting register
iqft = QFT(num_qubits=t, inverse=True, do_swaps=True)
qc.append(iqft, range(t))

# 5) Measure counting register
qc.measure(range(t), range(t))

qc.draw('mpl', fold=-1)

> Note: The controlled-phase emulation above is an instructional shortcut to study the QFT + phase-estimation step. 
> Replace it with true controlled modular multiplication gates for a full Shor implementation.

In [ ]:
backend = AerSimulator()
job = backend.run(qc, shots=4096)
result = job.result()
counts = result.get_counts()

print("Measurement counts:")
print(counts)

# Plot histogram manually to avoid extra imports
labels = list(counts.keys())
vals = [counts[k] for k in labels]

plt.figure(figsize=(8,4))
plt.bar(labels, vals)
plt.xlabel('Measured bitstring')
plt.ylabel('Counts')
plt.title('Counting-register outcomes')
plt.show()

## 4) Convert measurement to period candidate via continued fractions

If the measured integer is `m` (from `t` bits), we estimate:
\[
\phi \approx m / 2^t \approx s/r
\]
Then recover denominator `r` with `Fraction(...).limit_denominator(N)`.

In [ ]:
def bitstring_to_int(bitstr: str) -> int:
    # Qiskit returns classical bits in big-endian string form
    return int(bitstr, 2)

shots_total = sum(counts.values())
sorted_counts = sorted(counts.items(), key=lambda kv: kv[1], reverse=True)

candidates = []
for bitstr, c in sorted_counts[:6]:
    m = bitstring_to_int(bitstr)
    phase = m / (2**t)
    frac = Fraction(phase).limit_denominator(N)
    r = frac.denominator
    candidates.append((bitstr, c, phase, frac, r))

for bitstr, c, phase, frac, r in candidates:
    print(f"{bitstr} | count={c:4d} | phase={phase:.4f} ~ {frac} | candidate r={r}")

### Exercise 3

- Increase `t` to 5 or 6 and compare the quality of recovered `r`.
- Which measured bitstrings map to fractions with denominator 4?

## 5) Recover non-trivial factors from a valid even period

Given a candidate `r`:
1. `r` must be even.
2. `a^(r/2) \not\equiv -1 (mod N)`.
3. Compute:
\[
p = gcd(a^{r/2}-1, N), \quad q = gcd(a^{r/2}+1, N)
\]

In [ ]:
def try_factors_from_r(a, N, r):
    if r % 2 != 0:
        return None
    x = pow(a, r // 2, N)
    if x == N - 1:
        return None
    p = gcd(x - 1, N)
    q = gcd(x + 1, N)
    if 1 < p < N and 1 < q < N:
        return tuple(sorted((p, q)))
    return None

# test top candidate denominators
tested = sorted({r for *_, r in candidates})
print("Testing candidate r values:", tested)

for r in tested:
    factors = try_factors_from_r(a, N, r)
    print(f"r={r} -> factors={factors}")

## 6) Challenge tasks

1. Replace the emulated phase blocks with explicit controlled modular multiplication unitaries for `N=15`.
2. Generalize to another semiprime (small enough for simulation).
3. Compare shot counts (1024, 4096, 8192) and estimate success probability.
4. Add noise model simulation (optional) and observe robustness.

---
### Expected result for the default setup
For `N=15`, `a=2`, valid period is `r=4`, and the algorithm should yield factors `(3, 5)` from a good run.